In [22]:
import pathlib as Path
import numpy as np
import pandas as pd


## Create dataframe from embeddings

In [69]:
#-----Frequency Embeddings dataframe creation-----
def create_embeddings_dataframe(root_path):
    emb_root = Path.Path(root_path)
    emb_files = emb_root.rglob("embeddings_samples.npz")
    
    all_data = []
    
    for emb_file in emb_files:
        emb = np.load(emb_file)
        X = emb['X']
        y = emb['y']
        subjects = emb['subs']
        
        for i in range(X.shape[0]):
            data_point = {
                'embedding': X[i],
                'label': y[i],
                'subject': subjects[i],
                'file_path': str(emb_file)
            }
            all_data.append(data_point)
    
    df = pd.DataFrame(all_data)
    return df


## Clean embedding DF

In [70]:

def df_cleaning(df, emb_size=384):
        # --- Expand embedding column into 381 separate columns ---
    embedding_df = pd.DataFrame(df["embedding"].tolist(),
                                columns=[f"emb_{i}" for i in range(emb_size)])

    # --- Concatenate back to the original DataFrame (optional) ---
    df_expanded = pd.concat([df.drop(columns=["embedding"]), embedding_df], axis=1)
    return df_expanded

In [71]:
#-----Frequency Embeddings dataframe creation-----
freq_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//frequency_emb_stored//")
#-----Temporal Embeddings dataframe creation-----
temp_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//temporal_emb_stored//")
#-----Combined Embeddings dataframe creation-----
comb_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//emb_stored//")

freq_emb_df= df_cleaning(freq_emb, emb_size=384)
temp_emb_df= df_cleaning(temp_emb, emb_size=384)
comb_emb_df= df_cleaning(comb_emb, emb_size=768)

In [77]:
comb_emb_df = comb_emb_df.loc[~comb_emb_df.subject.duplicated(keep='first'), :]
temp_emb_df = temp_emb_df.loc[~temp_emb_df.subject.duplicated(keep='first'), :]
freq_emb_df = freq_emb_df.loc[~freq_emb_df.subject.duplicated(keep='first'), :]

# Metadata labeling
    - 0 -> HC
    - 1 -> unknown
    - 2 -> MCI-AD
    - 3 -> MCI- LBD

In [78]:
metadata_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//preDLB_shared(PSY_RAW).csv')
df_metadata = pd.read_csv(metadata_path, sep=";", encoding="utf-8-sig")

In [79]:
clinical_data_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//clinical_data_csv.csv')
df_clinical = pd.read_csv(clinical_data_path, sep=",", encoding="utf-8-sig")

In [80]:
handcrafted_features_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_hf//corpus_LBD_CZ_002_writing_results_table_original_filtered_extended.csv')
df_handcrafted_features = pd.read_csv(handcrafted_features_path, sep=";", encoding="utf-8-sig")

In [51]:
df_metadata

,ID_1.meranie,oficiálna dg,HC0_nHC1_MCI2_MCILB3_baseline,Vzdelani,Delka_vzdelani,JLO_HS,JLO_perc,Unnamed: 7,[1] JLO_Z,Vizuospacialni_funkce_Z,...,Unnamed: 149,CRT_Z.1,Premorbidni_Inteligence_Z.2,Verf_Lex_HS.2,Verf_Lex_Z.1,Verf_sem_HS.2,Verf_sem_Z.1,Razeni_obrazku_HS.2,Razeni_obrazku_Z.1,Exekutivni_funkce_Z.1
0,pre-LBD-1,NaN,1.0,3.0,13,26.0,56,"0,56","0,150969215","0,150969215",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,pre-LBD-2,NaN,3.0,3.0,13,27.0,72,"0,72","0,582841507","0,582841507",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pre-LBD-3,NaN,1.0,3.0,13,26.0,56,"0,56","0,150969215","0,150969215",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,pre-LBD-4,NaN,1.0,2.0,12,23.0,40,"0,4","-0,253347103","-0,253347103",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,pre-LBD-5,NaN,3.0,3.0,17,23.0,40,"0,4","-0,253347103","-0,253347103",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,pre-LBD-122,NaN,3.0,3.0,13,21.0,NaN,NaN,"-0,77",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
156,pre-LBD-124 (Harth),NaN,1.0,4.0,20,29.0,NaN,NaN,"1,08","0,54",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157,pre-LBD-125,NaN,2.0,3.0,13,23.0,40,NaN,"-0,25",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158,pre-LBD-126,NaN,2.0,3.0,13,30.0,86,"1,08","1,08032",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
df_clinical

,filename,#id,#personalID,MR ID,group,age,gender,LED,MOCA,education type,education length,memory z-score,visuo-spatial z-score,attention z-score,executive function z-score,GDS,pozn.
0,COBEN_COBEN_ACOUSTIC_VZ_AD01.wav,COBEN_CZCOBEN033,CZCOBEN033,1819A,aMCI,62.0,0.0,NaN,27.0,3.0,14,"0,22","1,08","-0,94","-1,95",6.0,roky education doplníme prumerem
1,COBEN_COBEN_ACOUSTIC_IK_AD02.wav,COBEN_CZCOBEN034,CZCOBEN034,1939A,aMCI,66.0,0.0,NaN,20.0,2.0,12,"-1,63","-0,77","-0,78","-1,95",0.0,NaN
2,COBEN_COBEN_ACOUSTIC_OS_AD03.wav,COBEN_CZCOBEN035,CZCOBEN035,2367A,aMCI,74.0,1.0,NaN,7.0,2.0,11,-3,"-1,34","-2,15","-1,51",4.0,NaN
3,COBEN_COBEN_ACOUSTIC_DV_AD04.wav,COBEN_CZCOBEN036,CZCOBEN036,2365A,aMCI,75.0,1.0,NaN,25.0,2.0,10,"-1,36","0,15","-0,38","-0,76",10.0,NaN
4,COBEN_COBEN_ACOUSTIC_HS_AD06.wav,COBEN_CZCOBEN038,CZCOBEN038,2305A,aMCI,69.0,1.0,NaN,22.0,2.0,12,"-0,15","-1,75","-1,03","-1,48",24.0,roky education doplníme prumerem
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,preDLB_pre-LBD-112.wav,preDLB_pre-LBD-112#1,pre-LBD-112#1,6009A,MCI-LB,84.0,1.0,NaN,24.0,3.0,13,"-0,69",NaN,-2,-1,7.0,NaN
220,preDLB_pre-LBD-114.wav,preDLB_pre-LBD-114#1,pre-LBD-114#1,6216A,MCI-LB,70.0,1.0,NaN,24.0,3.0,13,0,"-1,34","0,17","-0,56",8.0,NaN
221,preDLB_pre-LBD-120.wav,preDLB_pre-LBD-120#1,pre-LBD-120#1,6457A,MCI-LB,71.0,0.0,NaN,29.0,2.0,12,"-0,22","-0,25",0,"-1,61",7.0,NaN
222,preDLB_pre-LBD-115.wav,preDLB_pre-LBD-115#1,pre-LBD-115#1,5998A,PD,65.0,1.0,300,24.0,2.0,11,"-0,67","-2,17","-1,33","-1,17",11.0,NaN


In [81]:
import re
import pandas as pd

def append_col_when_main_contains_source(
    df_main, df_source, *, 
    match_col_main="subject",            # in df_main
    match_col_source="ID_1.meranie",     # in df_source
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + keep only rows in source with non-missing values
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()
    df_source = df_source.dropna(subset=[value_col])

    # init target column
    if new_col_name not in df_main:
        df_main[new_col_name] = pd.NA

    # for each source row, mark all main rows whose subject CONTAINS the source token
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token:
            continue
        mask = df_main[match_col_main].str.contains(re.escape(token), na=False, case=not case)
        # write only where we don't have a value yet (keeps first hit)
        to_set = mask & df_main[new_col_name].isna()
        df_main.loc[to_set, new_col_name] = r[value_col]

    return df_main

In [82]:
freq_emb_df_lbl = append_col_when_main_contains_source(
    df_main=freq_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [85]:

temp_emb_df_lbl = append_col_when_main_contains_source(
    df_main=temp_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [87]:

comb_emb_df_lbl = append_col_when_main_contains_source(
    df_main=comb_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)